# Data Information Saturation Analysis

**Goal:** Quantify whether useful learning signal (information content, diversity, co-occurrence richness) saturates or degrades as (a) member history length increases and (b) dataset size scales from 1.5M to 11M — stratified by frequency tier, age group, and line of business.

**Architecture:** Standalone analysis notebook that loads raw data from BigQuery at multiple scales, computes information-theoretic and distributional metrics at the member-level (within-member temporal saturation) and population-level (cross-member scaling saturation).

---
## R0: Technical Reference — Metrics Used in This Analysis

This section defines every metric used in the analysis, its mathematical formulation, why it was chosen, and how to interpret results.

### Shannon Entropy H(X)

**Formula:** $H(X) = -\sum_{x} p(x) \log_2 p(x)$

**What it measures:** The average number of bits needed to encode a draw from distribution $X$. Higher entropy = more uniform/diverse distribution; lower = more concentrated.

**Why chosen:** Entropy is the canonical information-theoretic measure of distributional diversity. Unlike variance, it is invariant to relabeling and naturally handles discrete categorical distributions (code frequencies). It directly quantifies "how much surprise" each observation carries.

**Interpretation:** For $N$ non-zero codes, $H_{max} = \log_2(N)$ (uniform). If $H \ll H_{max}$, the distribution is heavily concentrated on a few codes. In this analysis, flat entropy across data scales means the distributional shape is scale-invariant.

---

### Gini Coefficient

**Formula:** $G = \frac{2 \sum_{i=1}^{n} i \cdot x_{(i)}}{n \sum x_{(i)}} - \frac{n+1}{n}$ (from sorted values)

**What it measures:** Inequality in a non-negative distribution. $G=0$ means perfect equality (all codes equally frequent); $G=1$ means maximal inequality (one code dominates).

**Why chosen:** Gini is more sensitive to concentration in heavy-tailed distributions than entropy. Two distributions can have similar entropy but very different Gini coefficients if the tail shapes differ. Clinical code frequencies follow a power-law, making Gini the right complement to entropy.

**Interpretation:** $G > 0.9$ indicates extreme concentration — a small fraction of codes account for the vast majority of occurrences. Increasing Gini with scale means more data amplifies existing inequalities.

---

### Novelty Rate (Within-Member)

**Formula:** $\text{Novelty}(d) = \frac{|\{c \in \text{codes}(d) : c \notin \text{seen}(1..d{-}1)\}|}{|\text{codes}(d)|}$

**What it measures:** At day position $d$ in a member's history, what fraction of codes are genuinely new (never seen in days 1 through $d-1$)?

**Why chosen:** Directly answers "does later history add information?" for the transformer, which processes all 200 days via attention. If novelty drops to near-zero, later days are informationally redundant.

**Interpretation:** Novelty rate of 0.10 at day 50 means 90% of codes on day 50 are repeats of earlier days. The rate of decay indicates how quickly within-member information exhausts.

---

### Mutual Information I(A; B)

**Formula:** $I(A;B) = \sum_{a,b} p(a,b) \log_2 \frac{p(a,b)}{p(a) p(b)}$

**What it measures:** How much knowing code $A$'s presence/absence tells you about code $B$'s presence/absence. $I=0$ means independence; higher values mean stronger statistical association.

**Why chosen:** MI quantifies the relational structure between codes that the transformer's attention mechanism should exploit. If MI is near-zero between code pairs, codes are approximately conditionally independent — meaning a shared encoder has little relational structure to learn beyond marginal frequencies.

**Interpretation:** For binary variables (present/absent), max MI is ~1 bit (perfect correlation). Values $< 0.01$ bits indicate near-independence. The median MI across tier pairs indicates the "typical" relational signal available to the model.

---

### Conditional Entropy H(X_t | X_{t-1}, ..., X_1)

**Formula:** $H(X_t | X_{<t}) = H(X_1, ..., X_t) - H(X_1, ..., X_{t-1})$

**What it measures:** How much *new* information day $t$'s codes carry, given full knowledge of all previous days. This is the irreducible surprise at position $t$ — the information the model can extract from temporal sequencing.

**Why chosen:** Unlike novelty rate (which only measures set membership), conditional entropy captures the full distributional uncertainty reduction from temporal context. A code that appears 50% of the time on day 1 but 100% of the time after seeing the first 10 days has zero conditional entropy despite being "not novel."

**Interpretation:** Rapidly declining $H(X_t | X_{<t})$ means the temporal sequence becomes highly predictable early — the model learns less from later time steps. Flat conditional entropy means each day carries equal new information (ideal for sequence models).

---

### Jensen-Shannon Divergence JSD(P || Q)

**Formula:** $JSD(P \| Q) = \frac{1}{2} KL(P \| M) + \frac{1}{2} KL(Q \| M)$, where $M = \frac{P + Q}{2}$

**What it measures:** A symmetric, bounded measure of how different two probability distributions are. $JSD = 0$ means identical; $JSD = 1$ (in bits, base-2) means maximally different.

**Why chosen:** KL divergence is asymmetric and can be infinite. JSD is symmetric, always finite, and its square root is a proper metric. Used to compare code distributions across data scales.

**Interpretation:** $JSD < 0.001$ means distributions are effectively indistinguishable. In this analysis, JSD between adjacent data scales approaching zero proves the distribution shape is fixed regardless of dataset size.

---

### Co-occurrence Pair Entropy

**Formula:** Shannon entropy computed over the distribution of co-occurrence pair frequencies.

**What it measures:** How evenly distributed the co-occurrence signal is. High pair entropy = many diverse code-code relationships; low = signal concentrated on a few dominant pairs.

**Why chosen:** The transformer learns code relationships through attention. Pair entropy tells us whether the relational gradient signal is diverse (useful for learning rich representations) or concentrated (the model memorizes a few dominant co-occurrences).

**Interpretation:** Compare to $H_{max} = \log_2(\text{unique pairs})$. Ratio indicates effective relational diversity. Combined with pair Gini, identifies whether the model's attention is learning a broad or narrow set of relationships.

---

### Temporal Conditional Mutual Information I(A_t; B_t | history)

**Formula:** $I(A_t; B_t | C) = H(A_t | C) + H(B_t | C) - H(A_t, B_t | C)$, where $C$ represents the shared temporal history.

**What it measures:** Whether two codes on the same day share information *beyond* what the temporal history already predicts. This isolates the "fresh" relational signal at each time step.

**Why chosen:** Standard MI between code pairs conflates temporal autocorrelation with genuine cross-code dependence. Conditional MI removes the predictable component, revealing whether the model can learn something about code B from code A that it couldn't learn from the history alone.

**Interpretation:** If conditional MI ≈ unconditional MI, the history provides no shared context. If conditional MI ≪ unconditional MI, most apparent code association is explained by temporal patterns (chronic disease trajectories), not genuine code-code interaction.

---

### Trajectory Complexity Metrics

**Transition Entropy:** $H_{\text{trans}} = -\sum_{s'|s} p(s'|s) \log_2 p(s'|s)$ averaged over states $s$. Measures how predictable the next day's code set is given the current day's code set.

**Code Velocity:** $v(d) = |\text{codes}(d) \setminus \text{codes}(d{-}1)| + |\text{codes}(d{-}1) \setminus \text{codes}(d)|$. The symmetric difference between consecutive days' code sets — how much the clinical picture changes day-to-day.

**Trajectory Entropy Rate:** $h = \lim_{d \to \infty} H(X_d | X_{d-1}, ..., X_1)$. The asymptotic per-step information rate of the member's clinical trajectory — the fundamental limit on what a sequence model can learn per additional time step.

In [ ]:
import numpy as np
import pandas as pd
import json
import time
from collections import Counter, defaultdict
from scipy import stats
from scipy.spatial.distance import jensenshannon
from scipy.optimize import curve_fit
from google.cloud import bigquery
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# === Configuration ===
GCP_PROJECT = 'edp-prod-storage'
DATASET = 'edp-prod-storage.edp_ent_sdoheir_cns'

TABLES = {
    '1.5M': f'{DATASET}.a834793_Combined_All_LOB_o3_train_10pct_sample',
    'full': f'{DATASET}.a834793_Combined_All_LOB_o3_train_ending',
}

TARGET_CD_CNT = 6297
LEN_DY = 200
LEN_CD = 80

RAW_CD_VOCAB = 84_000

AGE_BUCKETS = {
    'pediatric': (0, 215),
    'young_adult': (216, 479),
    'middle_adult': (480, 779),
    'senior': (780, 1439),
}

RESULTS_DIR = Path('../../expe_analysis/exp_round5/data_saturation/')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

results = {}
print('Configuration loaded.')

In [ ]:
# === Data Loading Utilities ===

def load_sample_from_bigquery(table_name, sample_frac=None, min_dt_cnt=10,
                               max_rows=None, seed=42):
    """Load data from BigQuery with optional deterministic sampling."""
    client = bigquery.Client(project=GCP_PROJECT)

    where_clause = f"WHERE dt_cnt >= {min_dt_cnt}"
    if sample_frac and sample_frac < 1.0:
        where_clause += (
            f" AND MOD(ABS(FARM_FINGERPRINT("
            f"CAST(individual_id AS STRING))), 1000) < {int(sample_frac * 1000)}"
        )

    limit_clause = f"LIMIT {max_rows}" if max_rows else ""

    query = f"""
    SELECT individual_id, lob, age_in_months, cd, target, dt_cnt
    FROM `{table_name}`
    {where_clause}
    {limit_clause}
    """

    print(f"Loading from {table_name.split('.')[-1]}...")
    t0 = time.time()
    df = client.query(query).to_dataframe()
    print(f"  Loaded {len(df):,} rows in {time.time()-t0:.1f}s")
    return df


def parse_target_string(target_str):
    """Parse target string: '45,67*89*12,34' -> [[45,67], [89], [12,34]]"""
    if not target_str or pd.isna(target_str):
        return []
    days = target_str.split('*')[:LEN_DY]
    result = []
    for day_str in days:
        if not day_str:
            result.append([])
            continue
        codes = []
        for c in day_str.split(','):
            try:
                v = int(c)
                if 0 < v <= TARGET_CD_CNT:
                    codes.append(v)
            except (ValueError, TypeError):
                pass
        result.append(codes)
    return result


def parse_cd_string(cd_str):
    """Parse input code string into list of lists of ints."""
    if not cd_str or pd.isna(cd_str):
        return []
    days = cd_str.split('*')[:LEN_DY]
    result = []
    for day_str in days:
        if not day_str:
            result.append([])
            continue
        codes = []
        for c in day_str.split(','):
            try:
                v = int(c)
                if v > 0:
                    codes.append(v)
            except (ValueError, TypeError):
                pass
        result.append(codes)
    return result


def get_index_age(age_str):
    """Extract the age at index date (last non-zero value in age sequence)."""
    if not age_str or pd.isna(age_str):
        return 0
    parts = age_str.split('*')
    for p in reversed(parts):
        try:
            v = int(p)
            if v > 0:
                return min(v, 1439)
        except (ValueError, TypeError):
            continue
    return 0


def assign_age_bucket(age_months):
    for bucket, (lo, hi) in AGE_BUCKETS.items():
        if lo <= age_months <= hi:
            return bucket
    return 'unknown'


def compute_tier_boundaries(code_frequencies):
    """Compute tier boundaries using [20, 50, 80] percentiles on non-zero frequencies.
    Consistent with TierAwareTracker in moe_flashattn_4.py.
    """
    freq_nz = code_frequencies[code_frequencies > 0]
    if len(freq_nz) == 0:
        return {'p80': 0, 'p50': 0, 'p20': 0}
    percentiles = np.percentile(freq_nz, [20, 50, 80])
    return {
        'p80': percentiles[2],
        'p50': percentiles[1],
        'p20': percentiles[0],
    }


def assign_code_tier(code_idx, code_frequencies, tier_bounds):
    """Assign a 1-based code index to its frequency tier.
    Tiers match TierAwareTracker: common > p80, medium (p50, p80], rare (p20, p50], tail (0, p20].
    """
    if code_idx <= 0 or code_idx > len(code_frequencies):
        return 'unknown'
    freq = code_frequencies[code_idx - 1]
    if freq == 0:
        return 'zero'
    if freq > tier_bounds['p80']:
        return 'common'
    elif freq > tier_bounds['p50']:
        return 'medium'
    elif freq > tier_bounds['p20']:
        return 'rare'
    else:
        return 'tail'


def compute_code_frequencies(df, col='target'):
    """Compute target code frequency array from a DataFrame."""
    code_freq = np.zeros(TARGET_CD_CNT, dtype=np.int64)
    for target_str in df[col]:
        if not target_str or pd.isna(target_str):
            continue
        for day_str in target_str.split('*')[:LEN_DY]:
            if not day_str:
                continue
            for c_str in day_str.split(','):
                try:
                    v = int(c_str)
                    if 0 < v <= TARGET_CD_CNT:
                        code_freq[v - 1] += 1
                except (ValueError, TypeError):
                    pass
    return code_freq


def _gini(values):
    """Compute Gini coefficient of a non-negative array."""
    sorted_v = np.sort(values)
    n = len(sorted_v)
    if n == 0 or sorted_v.sum() == 0:
        return 0.0
    index = np.arange(1, n + 1)
    return float((2 * np.sum(index * sorted_v)) / (n * sorted_v.sum()) - (n + 1) / n)


def compute_cd_frequencies(df, top_n=10000):
    """Compute raw input code (cd) frequency Counter.
    Uses Counter (sparse) instead of dense array since cd vocab (~84k) is sparse.
    """
    freq = Counter()
    for cd_str in df['cd']:
        if not cd_str or pd.isna(cd_str):
            continue
        for day_str in cd_str.split('*')[:LEN_DY]:
            if not day_str:
                continue
            for c_str in day_str.split(','):
                try:
                    v = int(c_str)
                    if v > 0:
                        freq[v] += 1
                except (ValueError, TypeError):
                    pass
    return freq


def compute_cd_tier_boundaries(cd_freq_counter):
    """Compute tier boundaries for raw cd codes using the same
    percentile approach as target codes."""
    freqs = np.array(list(cd_freq_counter.values()))
    if len(freqs) == 0:
        return {'p80': 0, 'p50': 0, 'p20': 0}
    percentiles = np.percentile(freqs, [20, 50, 80])
    return {'p80': percentiles[2], 'p50': percentiles[1], 'p20': percentiles[0]}


def assign_cd_tier(code, cd_freq_counter, cd_tier_bounds):
    """Assign a raw cd code to its frequency tier."""
    freq = cd_freq_counter.get(code, 0)
    if freq == 0:
        return 'zero'
    if freq > cd_tier_bounds['p80']:
        return 'common'
    elif freq > cd_tier_bounds['p50']:
        return 'medium'
    elif freq > cd_tier_bounds['p20']:
        return 'rare'
    return 'tail'


print('Utilities defined.')

In [ ]:
# === Load 1.5M Dataset and Compute Frequencies ===

print("Loading 1.5M dataset...")
df_1_5m = load_sample_from_bigquery(TABLES['1.5M'], min_dt_cnt=10)
print(f"Columns: {list(df_1_5m.columns)}")
print(f"Shape: {df_1_5m.shape}")
print(f"LOB distribution:\n{df_1_5m['lob'].value_counts()}")

print("\nComputing target code frequencies...")
code_freq = compute_code_frequencies(df_1_5m, col='target')
tier_bounds = compute_tier_boundaries(code_freq)
print(f"Tier boundaries (on non-zero freqs): {tier_bounds}")
print(f"Non-zero codes: {(code_freq > 0).sum()} / {TARGET_CD_CNT}")
print(f"Total occurrences: {code_freq.sum():,}")

# Verify tier sizes match expected pattern
freq_nz = code_freq[code_freq > 0]
percentiles = np.percentile(freq_nz, [20, 50, 80])
common_mask = code_freq > percentiles[2]
medium_mask = (code_freq <= percentiles[2]) & (code_freq > percentiles[1])
rare_mask = (code_freq <= percentiles[1]) & (code_freq > percentiles[0])
tail_mask = (code_freq <= percentiles[0]) & (code_freq > 0)
print(f"\nTier sizes: common={common_mask.sum()}, medium={medium_mask.sum()}, "
      f"rare={rare_mask.sum()}, tail={tail_mask.sum()}")

print("\nComputing raw cd code frequencies (streaming Counter)...")
cd_freq = compute_cd_frequencies(df_1_5m)
cd_tier_bounds = compute_cd_tier_boundaries(cd_freq)
cd_vocab_size = len(cd_freq)
print(f"Raw cd vocabulary: {cd_vocab_size:,} unique codes")
print(f"Raw cd tier boundaries: {cd_tier_bounds}")
print(f"Total cd occurrences: {sum(cd_freq.values()):,}")

---
## Task 2: Within-Member Temporal Information Saturation

For each member, measure how much *new information* each successive day of their history contributes. Does the N-th day add as much novel signal as the 10th day?

In [ ]:
def compute_within_member_saturation(df, code_frequencies=None, tier_bounds=None,
                                     min_days=10, sample_n=50000, use_cd=False,
                                     cd_freq_counter=None, cd_tier_bounds=None):
    """
    For each member, walk through daily codes chronologically.
    At each day position d, measure:
      - cumulative unique codes seen so far
      - number of NEW codes on day d (not seen on days 1..d-1)
      - cumulative unique same-day co-occurrence PAIRS seen so far
      - number of NEW co-occurrence pairs on day d
    Stratified by tier, age bucket, LOB.
    Supports both target codes (default) and raw cd codes (use_cd=True).
    """
    if sample_n and len(df) > sample_n:
        df_sample = df.sample(n=sample_n, random_state=42)
    else:
        df_sample = df

    if use_cd and cd_freq_counter is not None:
        code_tier_map = {}
        for c in cd_freq_counter:
            code_tier_map[c] = assign_cd_tier(c, cd_freq_counter, cd_tier_bounds)
    else:
        code_tier_map = {}
        for i in range(1, TARGET_CD_CNT + 1):
            code_tier_map[i] = assign_code_tier(i, code_frequencies, tier_bounds)

    records = []
    t0 = time.time()
    processed = 0

    for idx, row in df_sample.iterrows():
        if use_cd:
            target_days = parse_cd_string(row['cd'])
        else:
            target_days = parse_target_string(row['target'])
        dt_cnt = int(row['dt_cnt'])
        n_days = min(dt_cnt, len(target_days))

        if n_days < min_days:
            continue

        age_months = get_index_age(row['age_in_months'])
        age_bucket = assign_age_bucket(age_months)
        lob = row['lob']

        seen_codes = set()
        seen_pairs = set()

        for d in range(n_days):
            day_codes = target_days[d]
            if not day_codes:
                continue

            new_codes = [c for c in day_codes if c not in seen_codes]

            sorted_dc = sorted(set(day_codes))
            n_dc = len(sorted_dc)
            day_pairs = set()
            for i in range(n_dc):
                for j in range(i + 1, min(n_dc, i + 20)):
                    day_pairs.add((sorted_dc[i], sorted_dc[j]))
            new_pairs = day_pairs - seen_pairs

            tier_new = defaultdict(int)
            for c in new_codes:
                tier = code_tier_map.get(c, 'unknown')
                if tier and tier not in ('zero', 'unknown'):
                    tier_new[tier] += 1

            seen_codes.update(day_codes)
            seen_pairs.update(day_pairs)

            records.append((
                d, n_days, lob, age_bucket,
                len(day_codes), len(new_codes), len(seen_codes),
                len(new_codes) / max(len(day_codes), 1),
                len(day_pairs), len(new_pairs), len(seen_pairs),
                len(new_pairs) / max(len(day_pairs), 1),
                tier_new.get('common', 0), tier_new.get('medium', 0),
                tier_new.get('rare', 0), tier_new.get('tail', 0),
            ))

        processed += 1
        if processed % 10000 == 0:
            print(f"  Processed {processed:,} members ({time.time()-t0:.0f}s)")

    columns = [
        'day_position', 'dt_cnt', 'lob', 'age_bucket',
        'n_codes_today', 'n_new_codes', 'cumul_unique_codes',
        'novelty_rate',
        'n_pairs_today', 'n_new_pairs', 'cumul_unique_pairs',
        'pair_novelty_rate',
        'new_common', 'new_medium', 'new_rare', 'new_tail',
    ]
    print(f"  Done. {processed:,} members, {len(records):,} rows in {time.time()-t0:.1f}s")
    return pd.DataFrame(records, columns=columns)

In [ ]:
print("Computing within-member saturation — target codes (50k sample)...")
within_member_df = compute_within_member_saturation(
    df_1_5m, code_freq, tier_bounds, min_days=10, sample_n=50000
)
print(f"Target result: {len(within_member_df):,} rows")

print("\nComputing within-member saturation — raw cd codes (20k sample)...")
within_member_cd_df = compute_within_member_saturation(
    df_1_5m, use_cd=True, cd_freq_counter=cd_freq, cd_tier_bounds=cd_tier_bounds,
    min_days=10, sample_n=20000
)
print(f"Raw cd result: {len(within_member_cd_df):,} rows")

In [ ]:
# === Within-Member Saturation Plots ===

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Within-Member Temporal Information Saturation', fontsize=14)

agg = within_member_df.groupby('day_position').agg(
    mean_novelty=('novelty_rate', 'mean'),
    mean_pair_novelty=('pair_novelty_rate', 'mean'),
    mean_cumul=('cumul_unique_codes', 'mean'),
).reset_index()

# (a) Overall novelty rate
ax = axes[0, 0]
ax.plot(agg['day_position'], agg['mean_novelty'], label='Target code novelty')
ax.plot(agg['day_position'], agg['mean_pair_novelty'], label='Target pair novelty', linestyle='--')
if len(within_member_cd_df) > 0:
    cd_agg = within_member_cd_df.groupby('day_position')['novelty_rate'].mean()
    ax.plot(cd_agg.index, cd_agg.values, label='Raw cd novelty', color='red', alpha=0.7)
ax.set_xlabel('Day position in history')
ax.set_ylabel('Novelty rate (fraction new)')
ax.set_title('Overall novelty rate by day position')
ax.legend()
ax.set_xlim(0, 200)

# (b) Cumulative unique codes
ax = axes[0, 1]
ax.plot(agg['day_position'], agg['mean_cumul'])
ax.set_xlabel('Day position')
ax.set_ylabel('Mean cumulative unique codes')
ax.set_title('Cumulative code discovery')

# (c) Novelty rate by tier
ax = axes[0, 2]
for tier in ['common', 'medium', 'rare', 'tail']:
    col_name = f'new_{tier}'
    tier_agg = within_member_df.groupby('day_position').agg(
        tier_sum=(col_name, 'sum'),
        codes_sum=('n_codes_today', 'sum'),
    )
    tier_novelty = tier_agg['tier_sum'] / tier_agg['codes_sum'].clip(lower=1)
    ax.plot(tier_novelty.index, tier_novelty.values, label=tier)
ax.set_xlabel('Day position')
ax.set_ylabel('Tier-specific novelty rate')
ax.set_title('Per-tier novelty decay')
ax.legend()

# (d) By LOB
ax = axes[1, 0]
for lob in ['Commercial', 'Medicare', 'Medicaid']:
    lob_df = within_member_df[within_member_df['lob'] == lob]
    if len(lob_df) == 0:
        continue
    lob_agg = lob_df.groupby('day_position')['novelty_rate'].mean()
    ax.plot(lob_agg.index, lob_agg.values, label=lob)
ax.set_xlabel('Day position')
ax.set_ylabel('Novelty rate')
ax.set_title('Novelty rate by LOB')
ax.legend()

# (e) By age bucket
ax = axes[1, 1]
for bucket in ['pediatric', 'young_adult', 'middle_adult', 'senior']:
    b_df = within_member_df[within_member_df['age_bucket'] == bucket]
    if len(b_df) == 0:
        continue
    b_agg = b_df.groupby('day_position')['novelty_rate'].mean()
    ax.plot(b_agg.index, b_agg.values, label=bucket)
ax.set_xlabel('Day position')
ax.set_ylabel('Novelty rate')
ax.set_title('Novelty rate by age group')
ax.legend()

# (f) Pair novelty
ax = axes[1, 2]
ax.plot(agg['day_position'], agg['mean_pair_novelty'])
ax.set_xlabel('Day position')
ax.set_ylabel('Co-occurrence pair novelty rate')
ax.set_title('Same-day pair novelty decay')

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'within_member_saturation.png'), dpi=150, bbox_inches='tight')
plt.show()

# Save numeric summary
within_member_summary = {
    'metric': 'within_member_temporal_saturation',
    'sample_size': int(within_member_df['dt_cnt'].nunique()),
}
for day_val in [10, 50, 100, 199]:
    key = f'novelty_rate_day_{day_val}'
    row = agg.loc[agg['day_position'] == day_val, 'mean_novelty']
    within_member_summary[key] = float(row.iloc[0]) if len(row) > 0 else None

results['within_member'] = within_member_summary
print(json.dumps(within_member_summary, indent=2))

within_member_cd_summary = {'metric': 'within_member_temporal_saturation_cd'}
if len(within_member_cd_df) > 0:
    cd_agg_full = within_member_cd_df.groupby('day_position')['novelty_rate'].mean().reset_index()
    for day_val in [10, 50, 100, 199]:
        key = f'novelty_rate_day_{day_val}'
        row = cd_agg_full.loc[cd_agg_full['day_position'] == day_val, 'novelty_rate']
        within_member_cd_summary[key] = float(row.iloc[0]) if len(row) > 0 else None
results['within_member_cd'] = within_member_cd_summary
print("\nRaw cd novelty:")
print(json.dumps(within_member_cd_summary, indent=2))

### How to Read Novelty Rate

For a given member at day position $d$, the notebook computes code novelty rate as:

$$\text{Novelty}(d) = \frac{\#\{\text{codes on day } d \text{ not seen on days } < d\}}{\#\{\text{codes on day } d\}}$$

Operationally, the implementation walks through each member history in order, keeps a running set of previously seen codes, counts how many codes on the current day are new relative to that earlier history, and divides by the total number of codes on that day. It then averages that fraction across members at each day position to produce the plotted curve.

Interpretation: if the novelty rate at a day position is 0.24, then on average 24% of codes at that position are new for that member and 76% are repeats of patterns already seen earlier in the timeline.

Figure description: the top-left panel shows novelty dropping sharply after the first few days; the top-middle panel shows cumulative unique code discovery still increasing but with a flattening slope; the top-right panel shows the same decay by frequency tier; the bottom-left and bottom-middle panels show that the pattern is similar across line of business and age groups; and the bottom-right panel shows that same-day code-pair novelty also declines over time.

---
## R1: Member Trajectory Analysis

Goes beyond point-in-time snapshots to characterize the *dynamics* of individual member clinical trajectories. Answers: How predictable are member trajectories? How much does the clinical picture change day-to-day? Do different trajectory types exist?

Computes for both **target codes** and **raw cd codes** in a single unified function.

In [ ]:
def compute_member_trajectory_analysis(df, code_freq, tier_bounds,
                                       sample_n=20000, use_cd=False):
    """
    Analyze member-level trajectory dynamics:
    - Code velocity: symmetric difference between consecutive days
    - Persistence score: Jaccard similarity across consecutive days
    - Transition entropy: diversity of state transitions
    - Trajectory type classification: stable/volatile/persistent/dynamic
    
    Operates on target codes by default; set use_cd=True for raw cd analysis.
    Memory-efficient: processes one member at a time, accumulates lightweight aggregates.
    """
    if sample_n and len(df) > sample_n:
        df_sample = df.sample(n=sample_n, random_state=42)
    else:
        df_sample = df

    parse_fn = parse_cd_string if use_cd else parse_target_string
    col = 'cd' if use_cd else 'target'
    label = 'cd' if use_cd else 'target'

    velocity_by_day = defaultdict(list)
    persistence_by_day = defaultdict(list)
    transition_counts = Counter()
    member_summaries = []

    t0 = time.time()
    processed = 0

    for _, row in df_sample.iterrows():
        days = parse_fn(row[col])
        dt_cnt = min(int(row['dt_cnt']), len(days))
        if dt_cnt < 5:
            continue

        lob = row['lob']
        age_bucket = assign_age_bucket(get_index_age(row['age_in_months']))

        velocities = []
        persistences = []
        prev_set = frozenset()

        for d in range(dt_cnt):
            curr_codes = days[d] if d < len(days) else []
            curr_set = frozenset(c for c in curr_codes if c > 0)

            if d > 0 and (prev_set or curr_set):
                added = len(curr_set - prev_set)
                removed = len(prev_set - curr_set)
                velocity = added + removed
                union_size = len(prev_set | curr_set)
                jaccard = len(prev_set & curr_set) / union_size if union_size > 0 else 1.0

                velocities.append(velocity)
                persistences.append(jaccard)
                velocity_by_day[d].append(velocity)
                persistence_by_day[d].append(jaccard)

                prev_hash = hash(prev_set) % 10000
                curr_hash = hash(curr_set) % 10000
                transition_counts[(prev_hash, curr_hash)] += 1

            prev_set = curr_set

        if velocities:
            mean_v = np.mean(velocities)
            std_v = np.std(velocities)
            mean_p = np.mean(persistences)

            if mean_v < 1.0:
                traj_type = 'stable'
            elif std_v / (mean_v + 1e-8) > 1.0:
                traj_type = 'volatile'
            elif mean_p > 0.8:
                traj_type = 'persistent'
            else:
                traj_type = 'dynamic'

            member_summaries.append({
                'lob': lob, 'age_bucket': age_bucket,
                'dt_cnt': dt_cnt,
                'mean_velocity': mean_v, 'std_velocity': std_v,
                'mean_persistence': mean_p,
                'trajectory_type': traj_type,
                'code_type': label,
            })

        processed += 1
        if processed % 5000 == 0:
            print(f"  [{label}] Processed {processed:,} ({time.time()-t0:.0f}s)")

    total_trans = sum(transition_counts.values())
    if total_trans > 0:
        trans_probs = np.array(list(transition_counts.values()), dtype=np.float64) / total_trans
        transition_entropy = float(stats.entropy(trans_probs, base=2))
    else:
        transition_entropy = 0.0

    day_positions = sorted(velocity_by_day.keys())
    velocity_curve = [(d, np.mean(velocity_by_day[d]), np.std(velocity_by_day[d]))
                      for d in day_positions if len(velocity_by_day[d]) >= 50]
    persistence_curve = [(d, np.mean(persistence_by_day[d]), np.std(persistence_by_day[d]))
                         for d in day_positions if len(persistence_by_day[d]) >= 50]

    summary_df = pd.DataFrame(member_summaries)

    print(f"  [{label}] Done: {processed:,} members, "
          f"transition entropy={transition_entropy:.2f} bits ({time.time()-t0:.0f}s)")

    return {
        'transition_entropy_bits': transition_entropy,
        'n_unique_transitions': len(transition_counts),
        'velocity_curve': velocity_curve,
        'persistence_curve': persistence_curve,
        'summary_df': summary_df,
        'code_type': label,
    }

In [ ]:
print("=" * 60)
print("MEMBER TRAJECTORY ANALYSIS")
print("=" * 60)

traj_target = compute_member_trajectory_analysis(
    df_1_5m, code_freq, tier_bounds, sample_n=20000, use_cd=False)
traj_cd = compute_member_trajectory_analysis(
    df_1_5m, code_freq, tier_bounds, sample_n=20000, use_cd=True)

fig, axes = plt.subplots(2, 4, figsize=(24, 10))
fig.suptitle('Member Trajectory Analysis', fontsize=14)

for col_offset, traj, label in [(0, traj_target, 'Target'), (2, traj_cd, 'Raw cd')]:
    sdf = traj['summary_df']

    ax = axes[0, col_offset]
    vc = traj['velocity_curve']
    if vc:
        days_v, means_v, stds_v = zip(*vc)
        ax.plot(days_v, means_v, label='Mean velocity')
        ax.fill_between(days_v,
                        np.array(means_v) - np.array(stds_v),
                        np.array(means_v) + np.array(stds_v), alpha=0.2)
    ax.set_xlabel('Day position')
    ax.set_ylabel('Code velocity (sym. diff)')
    ax.set_title(f'{label}: Day-to-day code velocity')

    ax = axes[0, col_offset + 1]
    pc = traj['persistence_curve']
    if pc:
        days_p, means_p, stds_p = zip(*pc)
        ax.plot(days_p, means_p, label='Mean Jaccard')
        ax.fill_between(days_p,
                        np.array(means_p) - np.array(stds_p),
                        np.array(means_p) + np.array(stds_p), alpha=0.2)
    ax.set_xlabel('Day position')
    ax.set_ylabel('Day-to-day Jaccard similarity')
    ax.set_title(f'{label}: Code persistence')

    ax = axes[1, col_offset]
    if len(sdf) > 0:
        type_counts = sdf['trajectory_type'].value_counts()
        type_counts.plot(kind='bar', ax=ax, color=['#2ecc71', '#e74c3c', '#3498db', '#f39c12'])
    ax.set_title(f'{label}: Trajectory type distribution')
    ax.set_ylabel('Count')

    ax = axes[1, col_offset + 1]
    if len(sdf) > 0:
        for lob in ['Commercial', 'Medicare', 'Medicaid']:
            lob_df = sdf[sdf['lob'] == lob]
            if len(lob_df) > 0:
                ax.hist(lob_df['mean_velocity'], bins=50, alpha=0.5, label=lob, density=True)
    ax.set_xlabel('Mean code velocity')
    ax.set_ylabel('Density')
    ax.set_title(f'{label}: Velocity distribution by LOB')
    ax.legend()

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'member_trajectory_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()

for traj, key in [(traj_target, 'trajectory_target'), (traj_cd, 'trajectory_cd')]:
    sdf = traj['summary_df']
    results[key] = {
        'transition_entropy_bits': traj['transition_entropy_bits'],
        'n_unique_transitions': traj['n_unique_transitions'],
        'mean_velocity': float(sdf['mean_velocity'].mean()) if len(sdf) > 0 else 0,
        'mean_persistence': float(sdf['mean_persistence'].mean()) if len(sdf) > 0 else 0,
        'trajectory_type_distribution': sdf['trajectory_type'].value_counts().to_dict() if len(sdf) > 0 else {},
        'by_lob': {
            lob: {
                'mean_velocity': float(g['mean_velocity'].mean()),
                'mean_persistence': float(g['mean_persistence'].mean()),
                'n': len(g),
            }
            for lob, g in sdf.groupby('lob') if len(g) > 0
        } if len(sdf) > 0 else {},
    }

print("\nTrajectory Summary (Target):")
print(json.dumps({k: v for k, v in results['trajectory_target'].items()
                  if k != 'by_lob'}, indent=2, default=str))
print("\nTrajectory Summary (Raw cd):")
print(json.dumps({k: v for k, v in results['trajectory_cd'].items()
                  if k != 'by_lob'}, indent=2, default=str))

---
## Task 3: Same-Day Co-occurrence and Temporal Transition Analysis

Measure co-occurrence pattern diversity (same-day code pairs + temporal skip-grams) and how it scales with data.

In [ ]:
def compute_cooccurrence_diversity(df, code_freq, tier_bounds,
                                   cd_freq_counter=None, cd_tier_bounds=None,
                                   sample_n=50000):
    """
    Compute same-day co-occurrence pair statistics and temporal skip-gram diversity.
    When cd_freq_counter is provided, also computes co-occurrence stats for raw
    input codes (cd column) in the same pass to avoid redundant iteration.
    """
    if sample_n and len(df) > sample_n:
        df_sample = df.sample(n=sample_n, random_state=42)
    else:
        df_sample = df

    same_day_pairs = Counter()
    temporal_bigrams = Counter()
    temporal_skipgrams_2 = Counter()
    temporal_skipgrams_3 = Counter()

    lob_pair_count = defaultdict(int)
    age_pair_count = defaultdict(int)

    code_tier_map = {}
    for i in range(1, TARGET_CD_CNT + 1):
        code_tier_map[i] = assign_code_tier(i, code_freq, tier_bounds)
    tier_pair_unique = defaultdict(set)

    include_cd = cd_freq_counter is not None and cd_tier_bounds is not None
    cd_same_day_pairs = Counter() if include_cd else None
    cd_temporal_bigrams = Counter() if include_cd else None
    cd_tier_pair_unique = defaultdict(set) if include_cd else None

    t0 = time.time()
    processed = 0

    for _, row in df_sample.iterrows():
        target_days = parse_target_string(row['target'])
        dt_cnt = min(int(row['dt_cnt']), len(target_days))
        lob = row['lob']
        age_bucket = assign_age_bucket(get_index_age(row['age_in_months']))

        cd_days = parse_cd_string(row['cd']) if include_cd else None

        for d in range(dt_cnt):
            day_codes = sorted(set(target_days[d])) if d < len(target_days) else []
            if day_codes:
                n_dc = len(day_codes)
                for i in range(min(n_dc, LEN_CD)):
                    for j in range(i + 1, min(n_dc, LEN_CD)):
                        pair = (day_codes[i], day_codes[j])
                        same_day_pairs[pair] += 1
                        lob_pair_count[lob] += 1
                        age_pair_count[age_bucket] += 1

                        tier_a = code_tier_map.get(day_codes[i], 'unknown')
                        tier_b = code_tier_map.get(day_codes[j], 'unknown')
                        tier_key = tuple(sorted([tier_a, tier_b]))
                        tier_pair_unique[tier_key].add(pair)

                day_sample = day_codes[:10]
                for skip, counter in [(1, temporal_bigrams),
                                      (2, temporal_skipgrams_2),
                                      (3, temporal_skipgrams_3)]:
                    if d + skip < dt_cnt and d + skip < len(target_days):
                        next_codes = target_days[d + skip][:10]
                        for a in day_sample:
                            for b in next_codes:
                                if a > 0 and b > 0:
                                    counter[(a, b)] += 1

            if include_cd and cd_days and d < len(cd_days):
                cd_day = sorted(set(cd_days[d]))[:30]
                n_cd = len(cd_day)
                for i in range(min(n_cd, 20)):
                    for j in range(i + 1, min(n_cd, 20)):
                        cd_same_day_pairs[(cd_day[i], cd_day[j])] += 1
                        tier_a = assign_cd_tier(cd_day[i], cd_freq_counter, cd_tier_bounds)
                        tier_b = assign_cd_tier(cd_day[j], cd_freq_counter, cd_tier_bounds)
                        cd_tier_pair_unique[tuple(sorted([tier_a, tier_b]))].add(
                            (cd_day[i], cd_day[j]))

                if d + 1 < dt_cnt and d + 1 < len(cd_days):
                    cd_src = cd_day[:10]
                    cd_tgt = sorted(set(cd_days[d + 1]))[:10]
                    for a in cd_src:
                        for b in cd_tgt:
                            if a > 0 and b > 0:
                                cd_temporal_bigrams[(a, b)] += 1

        processed += 1
        if processed % 10000 == 0:
            print(f"  Processed {processed:,} members ({time.time()-t0:.0f}s)")

    print(f"  Target: {len(same_day_pairs):,} unique same-day pairs")
    print(f"  Target: {len(temporal_bigrams):,} unique temporal bigrams")
    if include_cd:
        print(f"  Raw cd: {len(cd_same_day_pairs):,} unique same-day pairs")
        print(f"  Raw cd: {len(cd_temporal_bigrams):,} unique temporal bigrams")
    print(f"  Elapsed: {time.time()-t0:.0f}s")

    def distribution_entropy(counter):
        total = sum(counter.values())
        if total == 0:
            return 0.0
        probs = np.array(list(counter.values()), dtype=np.float64) / total
        return float(stats.entropy(probs, base=2))

    result = {
        'n_unique_same_day_pairs': len(same_day_pairs),
        'total_same_day_pair_occurrences': sum(same_day_pairs.values()),
        'same_day_pair_entropy_bits': distribution_entropy(same_day_pairs),
        'same_day_pair_gini': _gini(np.fromiter(
            same_day_pairs.values(), dtype=np.int64, count=len(same_day_pairs)
        )) if same_day_pairs else 0,
        'n_unique_temporal_bigrams': len(temporal_bigrams),
        'temporal_bigram_entropy_bits': distribution_entropy(temporal_bigrams),
        'n_unique_skip2grams': len(temporal_skipgrams_2),
        'skip2gram_entropy_bits': distribution_entropy(temporal_skipgrams_2),
        'n_unique_skip3grams': len(temporal_skipgrams_3),
        'skip3gram_entropy_bits': distribution_entropy(temporal_skipgrams_3),
        'tier_pair_diversity': {str(k): len(v) for k, v in tier_pair_unique.items()},
        'lob_pair_total_occurrences': dict(lob_pair_count),
        'age_pair_total_occurrences': dict(age_pair_count),
        'same_day_pair_concentration': {
            'top_10_pct_share': float(
                sum(sorted(same_day_pairs.values(), reverse=True)[:max(1, len(same_day_pairs) // 10)])
                / max(1, sum(same_day_pairs.values()))
            ),
            'top_1_pct_share': float(
                sum(sorted(same_day_pairs.values(), reverse=True)[:max(1, len(same_day_pairs) // 100)])
                / max(1, sum(same_day_pairs.values()))
            ),
        },
    }

    if include_cd:
        result['cd_cooccurrence'] = {
            'n_unique_same_day_pairs': len(cd_same_day_pairs),
            'total_same_day_pair_occurrences': sum(cd_same_day_pairs.values()),
            'same_day_pair_entropy_bits': distribution_entropy(cd_same_day_pairs),
            'same_day_pair_gini': _gini(np.fromiter(
                cd_same_day_pairs.values(), dtype=np.int64, count=len(cd_same_day_pairs)
            )) if cd_same_day_pairs else 0,
            'n_unique_temporal_bigrams': len(cd_temporal_bigrams),
            'temporal_bigram_entropy_bits': distribution_entropy(cd_temporal_bigrams),
            'tier_pair_diversity': {str(k): len(v) for k, v in cd_tier_pair_unique.items()},
            'cd_vocab_size': len(cd_freq_counter) if cd_freq_counter else 0,
        }

    return result

In [ ]:
print("Computing co-occurrence diversity on 1.5M (50k sample, target + raw cd)...")
cooccurrence_results_1_5m = compute_cooccurrence_diversity(
    df_1_5m, code_freq, tier_bounds,
    cd_freq_counter=cd_freq, cd_tier_bounds=cd_tier_bounds,
    sample_n=50000
)
results['cooccurrence_1_5m'] = cooccurrence_results_1_5m
print(json.dumps({k: v for k, v in cooccurrence_results_1_5m.items()
                  if k != 'cd_cooccurrence'}, indent=2, default=str))
if 'cd_cooccurrence' in cooccurrence_results_1_5m:
    print("\n--- Raw cd co-occurrence ---")
    print(json.dumps(cooccurrence_results_1_5m['cd_cooccurrence'], indent=2, default=str))

---
## Task 4: Target Code Distribution Shift Across Scales

Quantify how the *target* code distribution changes as dataset size grows, using nested FARM_FINGERPRINT sampling from the full table.

In [ ]:
def compute_target_distribution_at_scale(table_name, sample_fracs, min_dt_cnt=10,
                                         max_download_members=2_000_000):
    """
    Load incrementally larger subsets using deterministic FARM_FINGERPRINT sampling.
    For small scales, downloads full data. For large scales (> max_download_members),
    downloads a representative subsample and scales frequencies proportionally.
    """
    client = bigquery.Client(project=GCP_PROJECT)
    scale_results = []

    for frac in sample_fracs:
        frac_int = int(frac * 1000)
        t0 = time.time()

        count_query = f"""
        SELECT COUNT(*) as n
        FROM `{table_name}`
        WHERE dt_cnt >= {min_dt_cnt}
          AND MOD(ABS(FARM_FINGERPRINT(CAST(individual_id AS STRING))), 1000) < {frac_int}
        """
        n_members = int(client.query(count_query).to_dataframe().iloc[0]['n'])

        if n_members <= max_download_members:
            data_query = f"""
            SELECT target, lob, age_in_months, dt_cnt
            FROM `{table_name}`
            WHERE dt_cnt >= {min_dt_cnt}
              AND MOD(ABS(FARM_FINGERPRINT(CAST(individual_id AS STRING))), 1000) < {frac_int}
            """
            df_scale = client.query(data_query).to_dataframe()
            freq = compute_code_frequencies(df_scale, col='target')

            lob_counts = df_scale['lob'].value_counts().to_dict()
            df_scale['_age_idx'] = df_scale['age_in_months'].apply(get_index_age)
            df_scale['_age_bucket'] = df_scale['_age_idx'].apply(assign_age_bucket)
            age_counts = df_scale['_age_bucket'].value_counts().to_dict()
            del df_scale
        else:
            # Large scale: subsample, compute, then scale frequencies
            sub_frac = max(1, int(max_download_members / n_members * frac_int))
            sub_frac = min(sub_frac, frac_int)

            comp_query = f"""
            SELECT lob, age_in_months
            FROM `{table_name}`
            WHERE dt_cnt >= {min_dt_cnt}
              AND MOD(ABS(FARM_FINGERPRINT(CAST(individual_id AS STRING))), 1000) < {frac_int}
              AND MOD(ABS(FARM_FINGERPRINT(CAST(individual_id AS STRING))), {frac_int}) < {sub_frac}
            """
            df_comp = client.query(comp_query).to_dataframe()
            scale_factor = n_members / max(len(df_comp), 1)
            lob_counts = {k: int(v * scale_factor) for k, v in df_comp['lob'].value_counts().to_dict().items()}
            df_comp['_age_idx'] = df_comp['age_in_months'].apply(get_index_age)
            df_comp['_age_bucket'] = df_comp['_age_idx'].apply(assign_age_bucket)
            age_counts = {k: int(v * scale_factor) for k, v in df_comp['_age_bucket'].value_counts().to_dict().items()}
            del df_comp

            freq_query = f"""
            SELECT target
            FROM `{table_name}`
            WHERE dt_cnt >= {min_dt_cnt}
              AND MOD(ABS(FARM_FINGERPRINT(CAST(individual_id AS STRING))), 1000) < {frac_int}
              AND MOD(ABS(FARM_FINGERPRINT(CAST(individual_id AS STRING))), {frac_int}) < {sub_frac}
            """
            df_freq = client.query(freq_query).to_dataframe()
            freq_sample = compute_code_frequencies(df_freq, col='target')
            freq_scale = len(df_freq)
            del df_freq
            freq = (freq_sample.astype(np.float64) * (n_members / max(freq_scale, 1))).astype(np.int64)

        load_time = time.time() - t0

        freq_nz = freq[freq > 0]
        n_nonzero = len(freq_nz)
        gini = _gini(freq_nz) if n_nonzero > 0 else 0.0
        probs = freq_nz / freq_nz.sum() if freq_nz.sum() > 0 else np.array([1.0])
        entropy = float(stats.entropy(probs, base=2))

        tier_b = compute_tier_boundaries(freq)
        tier_coverage = {}
        for K in [10, 50, 100, 500]:
            for tier_name in ['common', 'medium', 'rare', 'tail']:
                if tier_name == 'common':
                    mask = freq > tier_b['p80']
                elif tier_name == 'medium':
                    mask = (freq > tier_b['p50']) & (freq <= tier_b['p80'])
                elif tier_name == 'rare':
                    mask = (freq > tier_b['p20']) & (freq <= tier_b['p50'])
                else:
                    mask = (freq > 0) & (freq <= tier_b['p20'])

                n_tier = mask.sum()
                n_above_K = ((freq >= K) & mask).sum()
                tier_coverage[f'{tier_name}_above_{K}'] = (
                    float(n_above_K / n_tier) if n_tier > 0 else 0
                )

        scale_results.append({
            'sample_frac': frac,
            'n_members': n_members,
            'n_nonzero_codes': int(n_nonzero),
            'total_occurrences': int(freq.sum()),
            'gini': float(gini),
            'entropy_bits': entropy,
            'max_frequency': int(freq.max()),
            'median_frequency': float(np.median(freq_nz)) if n_nonzero > 0 else 0,
            'tier_coverage': tier_coverage,
            'lob_counts': lob_counts,
            'age_counts': age_counts,
            'freq_array': freq,
        })

        print(f"  Scale {frac:.1%}: {n_members:,} members, "
              f"Gini={gini:.4f}, Entropy={entropy:.2f} bits, "
              f"nonzero={n_nonzero} (time={load_time:.0f}s)")

    return scale_results

In [ ]:
sample_fracs = [0.01, 0.02, 0.05, 0.10, 0.20, 0.40, 0.60, 0.80, 1.00]

print("Computing target distributions across scales...")
print("(Using nested FARM_FINGERPRINT sampling for monotonic inclusion)")
scale_results = compute_target_distribution_at_scale(
    TABLES['full'], sample_fracs, min_dt_cnt=5
)
results['cross_scale'] = [
    {k: v for k, v in r.items() if k != 'freq_array'}
    for r in scale_results
]

In [ ]:
def compute_cross_scale_divergences(scale_results):
    """Compute KL, JSD, marginal entropy gain between adjacent scales."""
    divergences = []
    for i in range(1, len(scale_results)):
        prev = scale_results[i - 1]
        curr = scale_results[i]

        p = prev['freq_array'].astype(np.float64)
        q = curr['freq_array'].astype(np.float64)

        eps = 1e-10
        p_norm = (p + eps) / (p + eps).sum()
        q_norm = (q + eps) / (q + eps).sum()

        kl_pq = float(stats.entropy(p_norm, q_norm, base=2))
        js = float(jensenshannon(p_norm, q_norm, base=2) ** 2)

        marginal_entropy = curr['entropy_bits'] - prev['entropy_bits']
        marginal_members = curr['n_members'] - prev['n_members']
        entropy_per_member = (
            marginal_entropy / marginal_members if marginal_members > 0 else 0
        )

        divergences.append({
            'from_frac': prev['sample_frac'],
            'to_frac': curr['sample_frac'],
            'from_members': prev['n_members'],
            'to_members': curr['n_members'],
            'marginal_members': marginal_members,
            'kl_divergence_bits': kl_pq,
            'js_divergence_bits': js,
            'marginal_entropy_gain_bits': marginal_entropy,
            'entropy_per_marginal_member': entropy_per_member,
            'gini_delta': curr['gini'] - prev['gini'],
        })

    return divergences

divergences = compute_cross_scale_divergences(scale_results)
results['cross_scale_divergences'] = divergences

for d in divergences:
    print(f"  {d['from_frac']:.0%} -> {d['to_frac']:.0%}: "
          f"JSD={d['js_divergence_bits']:.6f}, "
          f"dH={d['marginal_entropy_gain_bits']:.4f}, "
          f"dGini={d['gini_delta']:.6f}")

In [ ]:
# === Cross-Scale Distribution Plots ===

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Target Code Distribution Across Data Scales', fontsize=14)

members = [r['n_members'] for r in scale_results]

ax = axes[0, 0]
ax.plot(members, [r['gini'] for r in scale_results], 'o-')
ax.set_xlabel('Number of members')
ax.set_ylabel('Gini coefficient')
ax.set_title('Code frequency concentration vs scale')
ax.set_xscale('log')

ax = axes[0, 1]
ax.plot(members, [r['entropy_bits'] for r in scale_results], 'o-')
ax.set_xlabel('Number of members')
ax.set_ylabel('Shannon entropy (bits)')
ax.set_title('Target distribution entropy vs scale')
ax.set_xscale('log')

ax = axes[0, 2]
ax.plot(members, [r['n_nonzero_codes'] for r in scale_results], 'o-')
ax.set_xlabel('Number of members')
ax.set_ylabel('Non-zero target codes')
ax.set_title('Code coverage vs scale')
ax.set_xscale('log')
ax.axhline(y=TARGET_CD_CNT, color='r', linestyle='--', alpha=0.5, label=f'Max={TARGET_CD_CNT}')
ax.legend()

marg_members = [(d['from_members'] + d['to_members']) / 2 for d in divergences]

ax = axes[1, 0]
ax.plot(marg_members, [d['js_divergence_bits'] for d in divergences], 'o-')
ax.set_xlabel('Approx members (midpoint)')
ax.set_ylabel('JS divergence (bits)')
ax.set_title('Distribution shift between adjacent scales')
ax.set_xscale('log')

ax = axes[1, 1]
ax.plot(marg_members, [d['entropy_per_marginal_member'] for d in divergences], 'o-')
ax.set_xlabel('Approx members')
ax.set_ylabel('Entropy gain per marginal member')
ax.set_title('Diminishing information returns')
ax.set_xscale('log')
ax.set_yscale('log')

ax = axes[1, 2]
for tier in ['common', 'medium', 'rare', 'tail']:
    cov = [r['tier_coverage'][f'{tier}_above_100'] for r in scale_results]
    ax.plot(members, cov, 'o-', label=tier)
ax.set_xlabel('Number of members')
ax.set_ylabel('Fraction of codes with freq >= 100')
ax.set_title('Tier coverage (K=100) vs scale')
ax.set_xscale('log')
ax.legend()

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'cross_scale_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## R2.3: Temporal Conditional Entropy H(X_t | X_{t-1}, ..., X_1)

Measures how much *new* information each day position contributes given full knowledge of all prior days. This determines how much the model can actually learn from temporal sequences — the fundamental limit on per-step learning signal.

Unlike novelty rate (which only measures set membership), conditional entropy captures the full distributional uncertainty reduction. A code that was uncertain on day 1 but becomes certain by day 10 has low conditional entropy despite being "not novel."

Computed for both **target codes** and **raw cd codes**.

In [ ]:
def compute_temporal_conditional_entropy(df, sample_n=15000, max_days=100,
                                         use_cd=False, hash_bins=5000):
    """
    Estimate H(X_t | X_{<t}) for each day position t using discretized state
    representations.
    
    For each member, represent the "state at day d" as a hash of the cumulative
    code set up to day d. At each day position, build conditional frequency tables
    and compute: H(X_t | State) = H(X_t, State) - H(State).
    
    Memory: O(max_days * hash_bins) — bounded and predictable.
    """
    if sample_n and len(df) > sample_n:
        df_sample = df.sample(n=sample_n, random_state=42)
    else:
        df_sample = df

    parse_fn = parse_cd_string if use_cd else parse_target_string
    col = 'cd' if use_cd else 'target'
    label = 'cd' if use_cd else 'target'

    joint_counts = [Counter() for _ in range(max_days)]
    state_counts = [Counter() for _ in range(max_days)]
    marginal_counts = [Counter() for _ in range(max_days)]
    n_members_at_day = np.zeros(max_days, dtype=np.int64)

    t0 = time.time()
    processed = 0

    for _, row in df_sample.iterrows():
        days = parse_fn(row[col])
        dt_cnt = min(int(row['dt_cnt']), len(days), max_days)
        if dt_cnt < 3:
            continue

        cumul_codes = frozenset()
        for d in range(dt_cnt):
            curr_codes = tuple(sorted(set(c for c in (days[d] if d < len(days) else []) if c > 0)))
            if not curr_codes:
                continue

            state_hash = hash(cumul_codes) % hash_bins
            code_hash = hash(curr_codes) % hash_bins

            joint_counts[d][(state_hash, code_hash)] += 1
            state_counts[d][state_hash] += 1
            marginal_counts[d][code_hash] += 1
            n_members_at_day[d] += 1

            cumul_codes = frozenset(cumul_codes | set(curr_codes))

        processed += 1
        if processed % 5000 == 0:
            print(f"  [{label}] {processed:,} members ({time.time()-t0:.0f}s)")

    cond_entropy = []
    for d in range(max_days):
        n = n_members_at_day[d]
        if n < 100:
            continue
        total = sum(joint_counts[d].values())
        if total == 0:
            continue

        joint_probs = np.array(list(joint_counts[d].values()), dtype=np.float64) / total
        h_joint = float(stats.entropy(joint_probs, base=2))

        state_probs = np.array(list(state_counts[d].values()), dtype=np.float64) / total
        h_state = float(stats.entropy(state_probs, base=2))

        h_cond = max(0.0, h_joint - h_state)

        marginal_probs = np.array(list(marginal_counts[d].values()), dtype=np.float64) / total
        h_marginal = float(stats.entropy(marginal_probs, base=2))

        cond_entropy.append({
            'day': d,
            'H_cond': h_cond,
            'H_marginal': h_marginal,
            'reduction_ratio': 1.0 - (h_cond / h_marginal) if h_marginal > 0 else 0.0,
            'n_members': int(n),
        })

    print(f"  [{label}] Done: {len(cond_entropy)} day positions ({time.time()-t0:.0f}s)")
    return cond_entropy


print("Computing temporal conditional entropy...")
cond_ent_target = compute_temporal_conditional_entropy(
    df_1_5m, sample_n=15000, max_days=100, use_cd=False)
cond_ent_cd = compute_temporal_conditional_entropy(
    df_1_5m, sample_n=15000, max_days=100, use_cd=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Temporal Conditional Entropy H(X_t | X_{<t})', fontsize=14)

for cond_ent, label, color in [(cond_ent_target, 'Target', '#3498db'),
                                (cond_ent_cd, 'Raw cd', '#e74c3c')]:
    if not cond_ent:
        continue
    days = [r['day'] for r in cond_ent]
    h_cond = [r['H_cond'] for r in cond_ent]
    h_marg = [r['H_marginal'] for r in cond_ent]
    reduction = [r['reduction_ratio'] for r in cond_ent]

    axes[0].plot(days, h_cond, label=f'{label} H(X_t|past)', color=color)
    axes[0].plot(days, h_marg, label=f'{label} H(X_t)', color=color, linestyle='--', alpha=0.5)
    axes[1].plot(days, reduction, label=label, color=color)

axes[0].set_xlabel('Day position')
axes[0].set_ylabel('Entropy (bits)')
axes[0].set_title('Conditional vs marginal entropy')
axes[0].legend()
axes[1].set_xlabel('Day position')
axes[1].set_ylabel('Reduction ratio (1 - H_cond/H_marg)')
axes[1].set_title('Information reduction from temporal context')
axes[1].legend()

if cond_ent_target:
    late_days = [r for r in cond_ent_target if r['day'] >= 50]
    if late_days:
        entropy_rate = np.mean([r['H_cond'] for r in late_days])
        axes[2].axhline(y=entropy_rate, color='#3498db', linestyle='--',
                        label=f'Target rate: {entropy_rate:.3f}')
    days_t = [r['day'] for r in cond_ent_target]
    h_cond_t = [r['H_cond'] for r in cond_ent_target]
    axes[2].plot(days_t, h_cond_t, color='#3498db', alpha=0.5)
if cond_ent_cd:
    late_days_cd = [r for r in cond_ent_cd if r['day'] >= 50]
    if late_days_cd:
        entropy_rate_cd = np.mean([r['H_cond'] for r in late_days_cd])
        axes[2].axhline(y=entropy_rate_cd, color='#e74c3c', linestyle='--',
                        label=f'cd rate: {entropy_rate_cd:.3f}')
    days_cd = [r['day'] for r in cond_ent_cd]
    h_cond_cd = [r['H_cond'] for r in cond_ent_cd]
    axes[2].plot(days_cd, h_cond_cd, color='#e74c3c', alpha=0.5)
axes[2].set_xlabel('Day position')
axes[2].set_ylabel('H(X_t | past) bits')
axes[2].set_title('Entropy rate convergence')
axes[2].legend()

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'temporal_conditional_entropy.png'), dpi=150, bbox_inches='tight')
plt.show()

results['conditional_entropy_target'] = cond_ent_target
results['conditional_entropy_cd'] = cond_ent_cd
print(f"Target: {len(cond_ent_target)} day positions computed")
print(f"Raw cd: {len(cond_ent_cd)} day positions computed")
if cond_ent_target:
    late = [r['H_cond'] for r in cond_ent_target if r['day'] >= 50]
    if late:
        print(f"Target entropy rate (days 50+): {np.mean(late):.4f} bits")
if cond_ent_cd:
    late = [r['H_cond'] for r in cond_ent_cd if r['day'] >= 50]
    if late:
        print(f"Raw cd entropy rate (days 50+): {np.mean(late):.4f} bits")

---
## R2.4: Temporal Conditional Mutual Information — All Tier Pairs

Standard MI conflates chronic disease autocorrelation with genuine code-code interaction. Temporal conditional MI isolates the "fresh" relational signal at each time step by conditioning on shared history.

Computes **all** tier pairs: common-common, common-medium, common-rare, common-tail, medium-medium, medium-rare, medium-tail, rare-rare, rare-tail, tail-tail. For both target and raw cd codes.

In [ ]:
def compute_temporal_conditional_mi(df, code_freq, tier_bounds,
                                    cd_freq_counter=None, cd_tier_bounds=None,
                                    sample_n=15000, top_k=500):
    """
    Compute temporal conditional MI: I(A_t; B_t | history) for all tier pairs.
    
    For each pair (A, B), computes unconditional MI and MI conditioned on
    whether both codes were present in a 5-day lookback window. This separates
    chronic co-occurrence (autocorrelation) from genuine contemporaneous association.
    
    Processes target and cd codes in a single call for efficiency.
    """
    if sample_n and len(df) > sample_n:
        df_sample = df.sample(n=sample_n, random_state=42)
    else:
        df_sample = df

    all_results = {}
    code_configs = [('target', parse_target_string, 'target', code_freq, tier_bounds)]
    if cd_freq_counter is not None and cd_tier_bounds is not None:
        code_configs.append(('cd', parse_cd_string, 'cd', cd_freq_counter, cd_tier_bounds))

    for code_type, parse_fn, col, freq_data, tb in code_configs:
        if code_type == 'target':
            top_codes = np.argsort(freq_data)[-top_k:][::-1] + 1
            get_tier = lambda c, fd=freq_data, t=tb: assign_code_tier(int(c), fd, t)
        else:
            sorted_cd = sorted(freq_data.keys(), key=freq_data.get, reverse=True)[:top_k]
            top_codes = np.array(sorted_cd)
            get_tier = lambda c, fd=freq_data, t=tb: assign_cd_tier(int(c), fd, t)

        tier_assignments = {int(c): get_tier(c) for c in top_codes}

        tier_pairs_list = [
            ('common', 'common'), ('common', 'medium'), ('common', 'rare'),
            ('common', 'tail'), ('medium', 'medium'), ('medium', 'rare'),
            ('medium', 'tail'), ('rare', 'rare'), ('rare', 'tail'), ('tail', 'tail'),
        ]

        rng = np.random.RandomState(42)
        sampled_pairs_by_tier = {}

        for tier_a, tier_b in tier_pairs_list:
            codes_a = [int(c) for c in top_codes if tier_assignments.get(int(c)) == tier_a]
            codes_b = [int(c) for c in top_codes if tier_assignments.get(int(c)) == tier_b]
            if not codes_a or not codes_b:
                continue
            n_sample_pairs = min(50, len(codes_a) * len(codes_b))
            pairs = set()
            attempts = 0
            while len(pairs) < n_sample_pairs and attempts < n_sample_pairs * 5:
                a = codes_a[rng.randint(len(codes_a))]
                b = codes_b[rng.randint(len(codes_b))]
                if a != b:
                    pairs.add((a, b))
                attempts += 1
            sampled_pairs_by_tier[(tier_a, tier_b)] = list(pairs)

        WINDOW = 5
        pair_stats = {}
        for tier_key, pairs in sampled_pairs_by_tier.items():
            for a, b in pairs:
                pair_stats[(a, b)] = {
                    'n_both': 0, 'n_a_only': 0, 'n_b_only': 0, 'n_neither': 0,
                    'n_both_cond': 0, 'n_a_cond': 0, 'n_b_cond': 0, 'n_neither_cond': 0,
                    'n_hist_present': 0, 'n_hist_absent': 0,
                }

        t0 = time.time()
        processed = 0

        for _, row in df_sample.iterrows():
            days = parse_fn(row[col])
            dt_cnt = min(int(row['dt_cnt']), len(days))
            if dt_cnt < WINDOW + 1:
                continue

            day_presence = []
            for d in range(dt_cnt):
                curr = set(days[d]) if d < len(days) else set()
                day_presence.append(curr)

            for d in range(WINDOW, dt_cnt):
                history = set()
                for hd in range(max(0, d - WINDOW), d):
                    history.update(day_presence[hd])

                curr = day_presence[d]

                for (a, b), ps in pair_stats.items():
                    a_present = a in curr
                    b_present = b in curr
                    hist_has_both = a in history and b in history

                    if a_present and b_present:
                        ps['n_both'] += 1
                    elif a_present:
                        ps['n_a_only'] += 1
                    elif b_present:
                        ps['n_b_only'] += 1
                    else:
                        ps['n_neither'] += 1

                    if hist_has_both:
                        ps['n_hist_present'] += 1
                        if a_present and b_present:
                            ps['n_both_cond'] += 1
                        elif a_present:
                            ps['n_a_cond'] += 1
                        elif b_present:
                            ps['n_b_cond'] += 1
                        else:
                            ps['n_neither_cond'] += 1
                    else:
                        ps['n_hist_absent'] += 1

            processed += 1
            if processed % 3000 == 0:
                print(f"  [{code_type}] {processed:,} members ({time.time()-t0:.0f}s)")

        def compute_mi_from_counts(n11, n10, n01, n00):
            total = n11 + n10 + n01 + n00
            if total == 0:
                return 0.0
            p11, p10, p01, p00 = n11/total, n10/total, n01/total, n00/total
            pa = p11 + p10
            pb = p11 + p01
            mi = 0.0
            for pj, pm_a, pm_b in [(p11, pa, pb), (p10, pa, 1-pb),
                                     (p01, 1-pa, pb), (p00, 1-pa, 1-pb)]:
                if pj > 0 and pm_a > 0 and pm_b > 0:
                    mi += pj * np.log2(pj / (pm_a * pm_b))
            return max(0.0, mi)

        tier_results = []
        for tier_key, pairs in sampled_pairs_by_tier.items():
            mi_vals, cmi_vals = [], []
            for a, b in pairs:
                ps = pair_stats[(a, b)]
                mi = compute_mi_from_counts(ps['n_both'], ps['n_a_only'],
                                            ps['n_b_only'], ps['n_neither'])
                cmi = compute_mi_from_counts(ps['n_both_cond'], ps['n_a_cond'],
                                             ps['n_b_cond'], ps['n_neither_cond'])
                mi_vals.append(mi)
                cmi_vals.append(cmi)

            if mi_vals:
                tier_results.append({
                    'tier_a': tier_key[0], 'tier_b': tier_key[1],
                    'mean_mi': float(np.mean(mi_vals)),
                    'median_mi': float(np.median(mi_vals)),
                    'mean_cond_mi': float(np.mean(cmi_vals)),
                    'median_cond_mi': float(np.median(cmi_vals)),
                    'mean_reduction': float(1 - np.mean(cmi_vals) / max(np.mean(mi_vals), 1e-10)),
                    'n_pairs': len(mi_vals),
                })

        all_results[code_type] = tier_results
        print(f"  [{code_type}] Completed: {len(tier_results)} tier pairs ({time.time()-t0:.0f}s)")

    return all_results


print("Computing temporal conditional MI (all tier pairs, target + cd)...")
temporal_cmi = compute_temporal_conditional_mi(
    df_1_5m, code_freq, tier_bounds,
    cd_freq_counter=cd_freq, cd_tier_bounds=cd_tier_bounds,
    sample_n=15000, top_k=500
)
results['temporal_conditional_mi'] = temporal_cmi

for code_type, tier_results in temporal_cmi.items():
    print(f"\n--- {code_type.upper()} Temporal Conditional MI ---")
    print(f"{'Tier A':<10} {'Tier B':<10} {'MI':>10} {'Cond MI':>10} {'Reduction':>10} {'Pairs':>6}")
    for r in tier_results:
        print(f"{r['tier_a']:<10} {r['tier_b']:<10} "
              f"{r['mean_mi']:>10.6f} {r['mean_cond_mi']:>10.6f} "
              f"{r['mean_reduction']:>10.2%} {r['n_pairs']:>6}")

---
## Task 5: Conditional Entropy and Mutual Information Analysis

Measure how much information about code B is gained by knowing code A is present (mutual information), stratified by tier.

In [ ]:
def compute_conditional_information(df, code_freq, tier_bounds,
                                    sample_n=30000, top_k_codes=500):
    """
    Compute mutual information between code pairs using member-level
    binary presence vectors for the top_k most frequent codes.
    Vectorized computation on numpy arrays.
    """
    if sample_n and len(df) > sample_n:
        df_sample = df.sample(n=sample_n, random_state=42)
    else:
        df_sample = df

    tier_codes = defaultdict(list)
    for i in range(1, TARGET_CD_CNT + 1):
        t = assign_code_tier(i, code_frequencies, tier_bounds)
        if t not in ('zero', 'unknown'):
            tier_codes[t].append((code_frequencies[i-1], i))

    selected = []
    per_tier = top_k_codes // 4
    for tier in ['common', 'medium', 'rare', 'tail']:
        tier_sorted = sorted(tier_codes[tier], reverse=True)[:per_tier]
        selected.extend([c for _, c in tier_sorted])

    selected_set = set(selected)
    remaining = top_k_codes - len(selected)
    if remaining > 0:
        all_sorted = np.argsort(code_freq)[::-1] + 1
        for c in all_sorted:
            if int(c) not in selected_set:
                selected.append(int(c))
                if len(selected) >= top_k_codes:
                    break

    top_codes = np.array(selected)
    code_to_idx = {int(c): i for i, c in enumerate(top_codes)}

    n_members = len(df_sample)
    presence = np.zeros((n_members, top_k_codes), dtype=np.int8)

    t0 = time.time()
    for mem_idx, (_, row) in enumerate(df_sample.iterrows()):
        target_days = parse_target_string(row['target'])
        for day_codes in target_days:
            for c in day_codes:
                if c in code_to_idx:
                    presence[mem_idx, code_to_idx[c]] = 1
        if (mem_idx + 1) % 10000 == 0:
            print(f"  Built presence for {mem_idx+1:,}/{n_members:,} ({time.time()-t0:.0f}s)")

    p_a = presence.mean(axis=0)

    tier_assignments = {}
    for c in top_codes:
        tier_assignments[int(c)] = assign_code_tier(int(c), code_freq, tier_bounds)

    rng = np.random.RandomState(42)
    mi_results = []

    pair_types = [
        ('common', 'common'), ('common', 'medium'), ('common', 'rare'),
        ('common', 'tail'), ('medium', 'medium'), ('medium', 'rare'),
        ('rare', 'rare'), ('rare', 'tail'), ('tail', 'tail'),
    ]

    for tier_a, tier_b in pair_types:
        codes_a = [int(c) for c in top_codes if tier_assignments[int(c)] == tier_a]
        codes_b = [int(c) for c in top_codes if tier_assignments[int(c)] == tier_b]

        if not codes_a or not codes_b:
            continue

        n_pairs = min(100, len(codes_a) * len(codes_b))
        sampled_pairs = set()
        attempts = 0
        while len(sampled_pairs) < n_pairs and attempts < n_pairs * 3:
            a = codes_a[rng.randint(len(codes_a))]
            b = codes_b[rng.randint(len(codes_b))]
            if a != b:
                sampled_pairs.add((a, b))
            attempts += 1

        for a, b in sampled_pairs:
            idx_a = code_to_idx[a]
            idx_b = code_to_idx[b]

            col_a = presence[:, idx_a]
            col_b = presence[:, idx_b]

            p_11 = float(((col_a == 1) & (col_b == 1)).mean())
            p_10 = float(((col_a == 1) & (col_b == 0)).mean())
            p_01 = float(((col_a == 0) & (col_b == 1)).mean())
            p_00 = float(((col_a == 0) & (col_b == 0)).mean())

            mi = 0.0
            for p_joint, p_marg_a, p_marg_b in [
                (p_11, p_a[idx_a], p_a[idx_b]),
                (p_10, p_a[idx_a], 1 - p_a[idx_b]),
                (p_01, 1 - p_a[idx_a], p_a[idx_b]),
                (p_00, 1 - p_a[idx_a], 1 - p_a[idx_b]),
            ]:
                if p_joint > 0 and p_marg_a > 0 and p_marg_b > 0:
                    mi += p_joint * np.log2(p_joint / (p_marg_a * p_marg_b))

            mi_results.append({
                'tier_a': tier_a, 'tier_b': tier_b,
                'code_a': a, 'code_b': b,
                'mutual_info_bits': mi,
                'p_cooccur': p_11,
            })

    mi_df = pd.DataFrame(mi_results)

    tier_mi_summary = mi_df.groupby(['tier_a', 'tier_b']).agg(
        mean_mi=('mutual_info_bits', 'mean'),
        median_mi=('mutual_info_bits', 'median'),
        max_mi=('mutual_info_bits', 'max'),
        mean_cooccur=('p_cooccur', 'mean'),
        n_pairs=('mutual_info_bits', 'count'),
    ).reset_index()

    return {
        'tier_mi_summary': tier_mi_summary.to_dict('records'),
        'overall_mean_mi': float(mi_df['mutual_info_bits'].mean()),
        'overall_median_mi': float(mi_df['mutual_info_bits'].median()),
        'n_pairs_analyzed': len(mi_df),
    }

In [ ]:
print("Computing conditional information analysis (30k sample, top 500 codes)...")
cond_info = compute_conditional_information(
    df_1_5m, code_freq, tier_bounds, sample_n=30000, top_k_codes=500
)
results['conditional_information'] = cond_info

print(f"\nMutual Information by Tier Pair:")
print(f"{'Tier A':<12} {'Tier B':<12} {'Mean MI':<12} {'Med MI':<12} {'Mean CoOcc':<12}")
for r in cond_info['tier_mi_summary']:
    print(f"{r['tier_a']:<12} {r['tier_b']:<12} {r['mean_mi']:<12.6f} "
          f"{r['median_mi']:<12.6f} {r['mean_cooccur']:<12.4f}")

---
## Task 6: Marginal Member Information Contribution

Directly answer: as we add each additional member, how much new information does the dataset gain?

In [ ]:
def compute_marginal_member_information(df, sample_n=100000, n_increments=50):
    """
    Process members one at a time (in random order) and measure cumulative
    information metrics at regular checkpoints. Uses sets for code/pair tracking
    with bounded pair generation per day.
    """
    if sample_n and len(df) > sample_n:
        df_sample = df.sample(n=sample_n, random_state=42).reset_index(drop=True)
    else:
        df_sample = df.reset_index(drop=True)

    n_total = len(df_sample)
    increment_size = max(1, n_total // n_increments)
    checkpoints = list(range(increment_size, n_total + 1, increment_size))
    if checkpoints[-1] != n_total:
        checkpoints.append(n_total)

    cumul_code_freq = np.zeros(TARGET_CD_CNT, dtype=np.int64)
    cumul_codes_seen = set()
    cumul_pairs_seen = set()
    cumul_bigrams_seen = set()

    lob_codes = defaultdict(set)
    age_codes = defaultdict(set)

    curve = []
    checkpoint_idx = 0
    t0 = time.time()

    for mem_idx in range(n_total):
        row = df_sample.iloc[mem_idx]
        target_days = parse_target_string(row['target'])
        dt_cnt = min(int(row['dt_cnt']), len(target_days))
        lob = row['lob']
        age_bucket = assign_age_bucket(get_index_age(row['age_in_months']))

        for d in range(dt_cnt):
            day_codes = target_days[d] if d < len(target_days) else []

            for c in day_codes:
                if 0 < c <= TARGET_CD_CNT:
                    cumul_code_freq[c - 1] += 1
                    cumul_codes_seen.add(c)
                    lob_codes[lob].add(c)
                    age_codes[age_bucket].add(c)

            sorted_dc = sorted(set(c for c in day_codes if 0 < c <= TARGET_CD_CNT))
            n_dc = len(sorted_dc)
            for i in range(min(n_dc, 20)):
                for j in range(i + 1, min(n_dc, 20)):
                    cumul_pairs_seen.add((sorted_dc[i], sorted_dc[j]))

            if d + 1 < dt_cnt and d + 1 < len(target_days):
                src = [c for c in day_codes if 0 < c <= TARGET_CD_CNT][:10]
                tgt = [c for c in target_days[d + 1] if 0 < c <= TARGET_CD_CNT][:10]
                for a in src:
                    for b in tgt:
                        cumul_bigrams_seen.add((a, b))

        if checkpoint_idx < len(checkpoints) and mem_idx + 1 >= checkpoints[checkpoint_idx]:
            freq_nz = cumul_code_freq[cumul_code_freq > 0]
            entropy = float(stats.entropy(
                freq_nz / freq_nz.sum(), base=2
            )) if len(freq_nz) > 0 else 0

            curve.append({
                'n_members': mem_idx + 1,
                'unique_codes': len(cumul_codes_seen),
                'unique_pairs': len(cumul_pairs_seen),
                'unique_bigrams': len(cumul_bigrams_seen),
                'entropy_bits': entropy,
                'nonzero_codes': int(len(freq_nz)),
                'lob_code_counts': {k: len(v) for k, v in lob_codes.items()},
                'age_code_counts': {k: len(v) for k, v in age_codes.items()},
            })
            checkpoint_idx += 1

            if (mem_idx + 1) % (max(1, n_total // 5)) < increment_size:
                elapsed = time.time() - t0
                print(f"  {mem_idx+1:,}/{n_total:,}: "
                      f"codes={len(cumul_codes_seen)}, "
                      f"pairs={len(cumul_pairs_seen):,}, "
                      f"bigrams={len(cumul_bigrams_seen):,}, "
                      f"H={entropy:.2f} ({elapsed:.0f}s)")

    return curve

In [ ]:
print("Computing marginal member information curve (100k members)...")
marginal_curve = compute_marginal_member_information(
    df_1_5m, sample_n=100000, n_increments=50
)
results['marginal_member_curve'] = marginal_curve

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Marginal Member Information Contribution', fontsize=14)

members = [p['n_members'] for p in marginal_curve]

ax = axes[0, 0]
ax.plot(members, [p['unique_codes'] for p in marginal_curve], 'o-')
ax.set_xlabel('Members processed')
ax.set_ylabel('Cumulative unique codes')
ax.set_title('Code discovery curve')

ax = axes[0, 1]
ax.plot(members, [p['unique_pairs'] for p in marginal_curve], 'o-')
ax.set_xlabel('Members processed')
ax.set_ylabel('Cumulative unique co-occurrence pairs')
ax.set_title('Pair discovery curve')

ax = axes[0, 2]
ax.plot(members, [p['unique_bigrams'] for p in marginal_curve], 'o-')
ax.set_xlabel('Members processed')
ax.set_ylabel('Cumulative unique temporal bigrams')
ax.set_title('Temporal bigram discovery curve')

ax = axes[1, 0]
ax.plot(members, [p['entropy_bits'] for p in marginal_curve], 'o-')
ax.set_xlabel('Members processed')
ax.set_ylabel('Shannon entropy (bits)')
ax.set_title('Entropy accumulation curve')

ax = axes[1, 1]
if len(marginal_curve) > 1:
    marginal_codes = [
        marginal_curve[i]['unique_codes'] - marginal_curve[i-1]['unique_codes']
        for i in range(1, len(marginal_curve))
    ]
    marginal_members_inc = [
        marginal_curve[i]['n_members'] - marginal_curve[i-1]['n_members']
        for i in range(1, len(marginal_curve))
    ]
    codes_per_member = [
        c / m if m > 0 else 0
        for c, m in zip(marginal_codes, marginal_members_inc)
    ]
    ax.plot(members[1:], codes_per_member, 'o-')
ax.set_xlabel('Members processed')
ax.set_ylabel('New codes per marginal member')
ax.set_title('Marginal information gain (derivative)')

ax = axes[1, 2]
lobs_present = set()
for p in marginal_curve:
    lobs_present.update(p['lob_code_counts'].keys())
for lob in sorted(lobs_present):
    vals = [p['lob_code_counts'].get(lob, 0) for p in marginal_curve]
    ax.plot(members, vals, 'o-', label=lob)
ax.set_xlabel('Members processed')
ax.set_ylabel('Unique codes from LOB')
ax.set_title('Code discovery by LOB')
ax.legend()

plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'marginal_member_information.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === Saturation Point Estimation ===

def log_model(x, a, b):
    return a * np.log(x) + b

x = np.array([p['n_members'] for p in marginal_curve], dtype=np.float64)
y_codes = np.array([p['unique_codes'] for p in marginal_curve], dtype=np.float64)
y_pairs = np.array([p['unique_pairs'] for p in marginal_curve], dtype=np.float64)
y_entropy = np.array([p['entropy_bits'] for p in marginal_curve], dtype=np.float64)

saturation_estimates = {}
for name, y in [('codes', y_codes), ('pairs', y_pairs), ('entropy', y_entropy)]:
    try:
        popt, _ = curve_fit(log_model, x, y)
        saturation_point = 100 * x[0]
        r_squared = 1 - np.sum((y - log_model(x, *popt))**2) / np.sum((y - y.mean())**2)
        print(f"{name}: y = {popt[0]:.2f} * log(x) + {popt[1]:.2f}  (R^2={r_squared:.4f})")
        print(f"  Estimated 99% saturation at ~{saturation_point:,.0f} members")
        saturation_estimates[name] = {
            'a': float(popt[0]), 'b': float(popt[1]),
            'r_squared': float(r_squared),
            'saturation_99pct': float(saturation_point),
        }
    except Exception as e:
        print(f"{name}: fit failed: {e}")

results['saturation_estimates'] = saturation_estimates

---
## Task 7: LOB-Stratified and Age-Stratified Deep Dive

In [ ]:
def compute_stratified_saturation(df, code_freq, tier_bounds,
                                  stratify_col, sample_per_stratum=10000):
    """
    Run within-member saturation and co-occurrence diversity separately
    for each stratum (LOB or age bucket).
    """
    df_work = df.copy()
    if stratify_col == 'age_bucket':
        df_work['age_bucket'] = df_work['age_in_months'].apply(
            lambda x: assign_age_bucket(get_index_age(x))
        )

    strata = df_work[stratify_col].unique()
    stratum_results = {}

    for s in sorted(strata):
        df_s = df_work[df_work[stratify_col] == s]
        n_s = len(df_s)
        print(f"\n  Stratum '{s}': {n_s:,} members")

        if n_s < 100:
            print(f"    Skipping (too few members)")
            continue

        sample_n = min(sample_per_stratum, n_s)

        wm = compute_within_member_saturation(
            df_s, code_freq, tier_bounds,
            min_days=10, sample_n=sample_n
        )

        if len(wm) == 0:
            print(f"    No members with >= 10 days, skipping")
            continue

        wm_agg = wm.groupby('day_position').agg(
            mean_novelty=('novelty_rate', 'mean'),
            mean_pair_novelty=('pair_novelty_rate', 'mean'),
            mean_cumul=('cumul_unique_codes', 'mean'),
        ).reset_index()

        cooc = compute_cooccurrence_diversity(
            df_s, code_freq, tier_bounds,
            sample_n=sample_n
        )

        s_freq = compute_code_frequencies(
            df_s.head(sample_per_stratum), col='target'
        )
        s_freq_nz = s_freq[s_freq > 0]
        s_gini = _gini(s_freq_nz) if len(s_freq_nz) > 0 else 0
        s_entropy = float(stats.entropy(
            s_freq_nz / s_freq_nz.sum(), base=2
        )) if len(s_freq_nz) > 0 else 0

        def safe_lookup(df_agg, day, col):
            row = df_agg.loc[df_agg['day_position'] == day, col]
            return float(row.iloc[0]) if len(row) > 0 else None

        stratum_results[str(s)] = {
            'n_members': n_s,
            'n_nonzero_codes': int(len(s_freq_nz)),
            'gini': float(s_gini),
            'entropy_bits': s_entropy,
            'novelty_rate_day_10': safe_lookup(wm_agg, 10, 'mean_novelty'),
            'novelty_rate_day_50': safe_lookup(wm_agg, 50, 'mean_novelty'),
            'cooccurrence': {
                k: v for k, v in cooc.items()
                if k not in ['same_day_pair_concentration']
            },
        }

    return stratum_results

In [ ]:
print("=" * 60)
print("LOB-STRATIFIED ANALYSIS")
print("=" * 60)
lob_results = compute_stratified_saturation(
    df_1_5m, code_freq, tier_bounds,
    stratify_col='lob', sample_per_stratum=10000
)
results['lob_stratified'] = lob_results

print("\n" + "=" * 60)
print("AGE-STRATIFIED ANALYSIS")
print("=" * 60)
age_results = compute_stratified_saturation(
    df_1_5m, code_freq, tier_bounds,
    stratify_col='age_bucket', sample_per_stratum=10000
)
results['age_stratified'] = age_results

print("\n\nSTRATIFIED SUMMARY")
print(f"{'Stratum':<18} {'Members':>10} {'NonZero':>8} {'Gini':>8} "
      f"{'Entropy':>10} {'Nov@10':>8} {'Nov@50':>8}")
print("-" * 80)
for name, res_dict in [('LOB', lob_results), ('Age', age_results)]:
    for s, r in sorted(res_dict.items()):
        n10 = f"{r['novelty_rate_day_10']:.4f}" if r.get('novelty_rate_day_10') is not None else 'N/A'
        n50 = f"{r['novelty_rate_day_50']:.4f}" if r.get('novelty_rate_day_50') is not None else 'N/A'
        print(f"{s:<18} {r['n_members']:>10,} {r['n_nonzero_codes']:>8} "
              f"{r['gini']:>8.4f} {r['entropy_bits']:>10.2f} "
              f"{n10:>8} {n50:>8}")

---
## Task 8: Marginal Member Characterization (Core vs Marginal)

In [ ]:
def compare_core_vs_marginal_members(table_full, core_frac=0.10,
                                     min_dt_cnt=5, sample_n=30000):
    """
    Compare 'core' members (in the smallest sample) vs. 'marginal' members
    (those added when scaling up). Measures whether marginal members are
    systematically less diverse.
    """
    client = bigquery.Client(project=GCP_PROJECT)
    core_threshold = int(core_frac * 1000)

    core_query = f"""
    SELECT target, lob, age_in_months, dt_cnt
    FROM `{table_full}`
    WHERE dt_cnt >= {min_dt_cnt}
      AND MOD(ABS(FARM_FINGERPRINT(CAST(individual_id AS STRING))), 1000) < {core_threshold}
    """

    # For marginal, use secondary hash filter + LIMIT to avoid downloading millions
    marginal_limit_query = f"""
    SELECT target, lob, age_in_months, dt_cnt
    FROM `{table_full}`
    WHERE dt_cnt >= {min_dt_cnt}
      AND MOD(ABS(FARM_FINGERPRINT(CAST(individual_id AS STRING))), 1000) >= {core_threshold}
      AND MOD(ABS(FARM_FINGERPRINT(CAST(individual_id AS STRING))), 10000) < {core_threshold + 10}
    LIMIT {sample_n}
    """

    print("Loading core members...")
    df_core = client.query(core_query).to_dataframe()
    if len(df_core) > sample_n:
        df_core = df_core.sample(n=sample_n, random_state=42)
    print(f"  Core: {len(df_core):,} members")

    print("Loading marginal members (sampled)...")
    df_marginal = client.query(marginal_limit_query).to_dataframe()
    if len(df_marginal) > sample_n:
        df_marginal = df_marginal.sample(n=sample_n, random_state=42)
    print(f"  Marginal: {len(df_marginal):,} members")

    def member_stats(df):
        stats_list = []
        for _, row in df.iterrows():
            target_days = parse_target_string(row['target'])
            dt_cnt = min(int(row['dt_cnt']), len(target_days))
            all_codes = set()
            total_codes = 0
            n_pairs = 0

            for d in range(dt_cnt):
                day_codes = target_days[d] if d < len(target_days) else []
                valid = [c for c in day_codes if 0 < c <= TARGET_CD_CNT]
                all_codes.update(valid)
                total_codes += len(valid)
                n_uniq = len(set(valid))
                n_pairs += n_uniq * (n_uniq - 1) // 2

            stats_list.append({
                'dt_cnt': dt_cnt,
                'unique_codes': len(all_codes),
                'approx_pairs': n_pairs,
                'total_occurrences': total_codes,
                'codes_per_day': total_codes / max(dt_cnt, 1),
                'code_density': len(all_codes) / max(dt_cnt, 1),
                'lob': row['lob'],
                'age_bucket': assign_age_bucket(get_index_age(row['age_in_months'])),
            })
        return pd.DataFrame(stats_list)

    print("Computing core member stats...")
    t0 = time.time()
    core_stats = member_stats(df_core)
    print(f"  Done in {time.time()-t0:.0f}s")

    print("Computing marginal member stats...")
    t0 = time.time()
    marginal_stats = member_stats(df_marginal)
    print(f"  Done in {time.time()-t0:.0f}s")

    comparison = {}
    for col in ['dt_cnt', 'unique_codes', 'approx_pairs', 'total_occurrences',
                'codes_per_day', 'code_density']:
        mw_stat, mw_p = stats.mannwhitneyu(
            core_stats[col], marginal_stats[col], alternative='two-sided'
        )
        comparison[col] = {
            'core_mean': float(core_stats[col].mean()),
            'core_median': float(core_stats[col].median()),
            'marginal_mean': float(marginal_stats[col].mean()),
            'marginal_median': float(marginal_stats[col].median()),
            'mannwhitney_p': float(mw_p),
            'effect_direction': (
                'core > marginal' if core_stats[col].mean() > marginal_stats[col].mean()
                else 'marginal > core'
            ),
        }

    comparison['lob_composition'] = {
        'core': core_stats['lob'].value_counts(normalize=True).to_dict(),
        'marginal': marginal_stats['lob'].value_counts(normalize=True).to_dict(),
    }
    comparison['age_composition'] = {
        'core': core_stats['age_bucket'].value_counts(normalize=True).to_dict(),
        'marginal': marginal_stats['age_bucket'].value_counts(normalize=True).to_dict(),
    }

    return comparison

In [ ]:
print("Comparing core vs marginal members...")
core_vs_marginal = compare_core_vs_marginal_members(
    TABLES['full'], core_frac=0.10, min_dt_cnt=5, sample_n=30000
)
results['core_vs_marginal'] = core_vs_marginal

print("\nCORE vs MARGINAL MEMBER COMPARISON")
print(f"{'Metric':<22} {'Core Mean':>12} {'Marginal Mean':>14} {'Direction':>20} {'p-value':>12}")
print("-" * 82)
for col, v in core_vs_marginal.items():
    if isinstance(v, dict) and 'core_mean' in v:
        print(f"{col:<22} {v['core_mean']:>12.2f} {v['marginal_mean']:>14.2f} "
              f"{v['effect_direction']:>20} {v['mannwhitney_p']:>12.2e}")

---
## Task 9: Save Results, Generate Report, and Final Synthesis

In [ ]:
# === Save all results to JSON ===

results_serializable = json.loads(json.dumps(results, default=str))

output_path = Path('../../expe_logs/exp_round5/data_information_saturation_results.json')
output_path.parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w') as f:
    json.dump(results_serializable, f, indent=2)
print(f"Results saved to {output_path}")

In [ ]:
# === Generate Markdown Report ===

report = []
report.append("# Data Information Saturation Analysis Results Report\n")
report.append(f"**Generated:** {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')}\n")
report.append("**Question:** Does useful learning signal saturate or degrade as "
              "(a) member history length increases and (b) dataset size scales?\n")
report.append("---\n")

report.append("## 1. Within-Member Temporal Saturation\n")
wm = results.get('within_member', {})
for key in ['novelty_rate_day_10', 'novelty_rate_day_50',
            'novelty_rate_day_100', 'novelty_rate_day_199']:
    val = wm.get(key, 'N/A')
    if isinstance(val, float):
        val = f"{val:.4f}"
    label = key.replace('novelty_rate_day_', 'Day ')
    report.append(f"- Novelty rate at {label}: **{val}**\n")
report.append("\n**Interpretation:** [Fill after reviewing plots]\n")

report.append("\n## 2. Cross-Scale Distribution Analysis\n")
if 'cross_scale_divergences' in results:
    report.append("| From | To | JS Div | Delta Entropy | Delta Gini |\n")
    report.append("|------|----|---------|-----------|---------| \n")
    for d in results['cross_scale_divergences']:
        report.append(
            f"| {d['from_frac']:.0%} | {d['to_frac']:.0%} | "
            f"{d['js_divergence_bits']:.6f} | "
            f"{d['marginal_entropy_gain_bits']:.4f} | "
            f"{d['gini_delta']:.6f} |\n"
        )

report.append("\n## 3. LOB-Stratified Results\n")
if 'lob_stratified' in results:
    report.append("| LOB | Members | Gini | Entropy | Nov@10 |\n")
    report.append("|-----|---------|------|---------|--------|\n")
    for s, r in sorted(results.get('lob_stratified', {}).items()):
        n10 = f"{r['novelty_rate_day_10']:.4f}" if r.get('novelty_rate_day_10') is not None else 'N/A'
        report.append(
            f"| {s} | {r['n_members']:,} | {r['gini']:.4f} | "
            f"{r['entropy_bits']:.2f} | {n10} |\n"
        )

report.append("\n## 4. Age-Stratified Results\n")
if 'age_stratified' in results:
    report.append("| Age Bucket | Members | Gini | Entropy | Nov@10 |\n")
    report.append("|------------|---------|------|---------|--------|\n")
    for s, r in sorted(results.get('age_stratified', {}).items()):
        n10 = f"{r['novelty_rate_day_10']:.4f}" if r.get('novelty_rate_day_10') is not None else 'N/A'
        report.append(
            f"| {s} | {r['n_members']:,} | {r['gini']:.4f} | "
            f"{r['entropy_bits']:.2f} | {n10} |\n"
        )

report.append("\n## 5. Core vs Marginal Members\n")
if 'core_vs_marginal' in results:
    report.append("| Metric | Core Mean | Marginal Mean | Direction | p-value |\n")
    report.append("|--------|-----------|---------------|-----------|--------|\n")
    for col, v in results.get('core_vs_marginal', {}).items():
        if isinstance(v, dict) and 'core_mean' in v:
            report.append(
                f"| {col} | {v['core_mean']:.2f} | {v['marginal_mean']:.2f} | "
                f"{v['effect_direction']} | {v['mannwhitney_p']:.2e} |\n"
            )

report.append("\n## 6. Conditional Information (Mutual Information by Tier)\n")
if 'conditional_information' in results:
    ci = results['conditional_information']
    report.append(f"- Overall mean MI: **{ci['overall_mean_mi']:.6f}** bits\n")
    report.append(f"- Pairs analyzed: **{ci['n_pairs_analyzed']}**\n")
    if 'tier_mi_summary' in ci:
        report.append("\n| Tier A | Tier B | Mean MI | Median MI | Pairs |\n")
        report.append("|--------|--------|---------|-----------|-------|\n")
        for r in ci['tier_mi_summary']:
            report.append(f"| {r['tier_a']} | {r['tier_b']} | "
                          f"{r['mean_mi']:.6f} | {r['median_mi']:.6f} | "
                          f"{r['n_pairs']} |\n")

report.append("\n## R1: Member Trajectory Analysis\n")
for key in ['trajectory_target', 'trajectory_cd']:
    if key in results:
        t = results[key]
        label = 'Target' if 'target' in key else 'Raw cd'
        report.append(f"\n### {label} Codes\n")
        report.append(f"- Transition entropy: **{t.get('transition_entropy_bits', 0):.2f}** bits\n")
        report.append(f"- Mean velocity: **{t.get('mean_velocity', 0):.2f}**\n")
        report.append(f"- Mean persistence (Jaccard): **{t.get('mean_persistence', 0):.3f}**\n")
        if 'trajectory_type_distribution' in t:
            report.append(f"- Trajectory types: {t['trajectory_type_distribution']}\n")

report.append("\n## R2.3: Temporal Conditional Entropy\n")
for key, label in [('conditional_entropy_target', 'Target'), ('conditional_entropy_cd', 'Raw cd')]:
    if key in results and results[key]:
        late = [r for r in results[key] if r['day'] >= 50]
        if late:
            rate = np.mean([r['H_cond'] for r in late])
            report.append(f"- **{label}** entropy rate (days 50+): **{rate:.4f}** bits\n")

report.append("\n## R2.4: Temporal Conditional MI (All Pairs)\n")
if 'temporal_conditional_mi' in results:
    for ct, tier_results in results['temporal_conditional_mi'].items():
        report.append(f"\n### {ct.upper()}\n")
        report.append("| Tier A | Tier B | MI | Cond MI | Reduction |\n")
        report.append("|--------|--------|-----|---------|----------|\n")
        for r in tier_results:
            report.append(f"| {r['tier_a']} | {r['tier_b']} | "
                          f"{r['mean_mi']:.6f} | {r['mean_cond_mi']:.6f} | "
                          f"{r['mean_reduction']:.2%} |\n")

report.append("\n## 7. Saturation Estimates\n")
if 'saturation_estimates' in results:
    for name, est in results['saturation_estimates'].items():
        if isinstance(est, dict) and 'r_squared' in est:
            report.append(f"- **{name}**: y = {est['a']:.2f} * log(x) + {est['b']:.2f} "
                          f"(R^2={est['r_squared']:.4f}), "
                          f"99% saturation at ~{est['saturation_99pct']:,.0f} members\n")

report.append("\n## 8. Conclusions\n")
report.append("\n[To be filled based on evidence above]\n")

report_text = ''.join(report)
report_path = RESULTS_DIR / 'data_information_saturation_report.md'
with open(report_path, 'w') as f:
    f.write(report_text)
print(f"Report saved to {report_path}")
print("\n" + "=" * 60)
print("ANALYSIS COMPLETE")
print("=" * 60)